# IHRAS — 02. Track B (GPU: L4, high RAM): vision jobs, external validation, official split, timing

Run **at the same time** as `01_track_a.ipynb` (two GPU sessions), with the same `RUN_ID`.
Then run `03_merge.ipynb`.

**Runtime:** L4 GPU with *High RAM* (more vCPUs: image decoding is the bottleneck).

| Job | What it computes | Output in `runs/<RUN_ID>/jobs/` |
|---|---|---|
| `B1_explain` | Grad-CAM, Grad-CAM++, segmentation, fusion on the 984 NIH boxes (GPU) | `xai_per_box.csv` |
| `B2_predict_models` | predictions of DenseNet (multi), ResNet-50 and 4 external models | `predictions_*.npz` |
| `B3_robustness` | noise and photometric perturbations | robustness keys |
| `B4_official` | official NIH test split (25,596 images), NIH and external models | `predictions_*__official_test.npz` |
| `B5_timing` | per-item time on this GPU, run alone at the end | `timing_per_item.csv` |

## 1. Configuration

In [ ]:
DRIVE_ROOT = "/content/drive/MyDrive/pesquisa/ihras2"   # project folder in Google Drive
RUN_ID = "run03"        # MUST be the same in the track A, track B and merge notebooks
TRACK = "B"

EXTERNAL = "densenet_pc,densenet_chex,densenet_mimic_ch,densenet_mimic_nb"   # weights trained without NIH
LLM_PUBLISHED = "deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B"                # timed in B5

## 2. Setup: Drive, code, dependencies, environment record, job runner

In [ ]:
# ---- Drive, paths, GPU -------------------------------------------------------
import datetime, glob, hashlib, json, os, platform, shlex, shutil, subprocess, sys, time, zipfile
from google.colab import drive
drive.mount("/content/drive")


def utc_now():
    """Current UTC time (ISO 8601), used in every log and provenance record."""
    return datetime.datetime.now(datetime.timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")


SESSION = datetime.datetime.now(datetime.timezone.utc).strftime("%Y%m%d_%H%M%S")
RUN = f"{DRIVE_ROOT}/runs/{RUN_ID}"
JOBS, LOGS = f"{RUN}/jobs", f"{RUN}/logs"           # one folder per job; logs and .done markers
ENV = f"{RUN}/env/track{TRACK}_session_{SESSION}"    # environment record of this session
CODE_SRC, CODE = f"{DRIVE_ROOT}/code", "/content/ihras_code"
DATA_DIR = f"{DRIVE_ROOT}/data/nih"                  # written by 00_data_setup.ipynb
NIH = "/content/nih"                                 # local images and metadata
LOCAL_ZIP = "/content/nih_data.zip"
for d in (JOBS, LOGS, ENV, NIH):
    os.makedirs(d, exist_ok=True)

import torch
GPU = torch.cuda.get_device_name(0) if torch.cuda.is_available() else None
DEVICE = "cuda" if GPU else "cpu"
WORKERS = max(2, (os.cpu_count() or 2) - 1)          # parallel image loaders
print(f"RUN_ID {RUN_ID} | track {TRACK} | session {SESSION} | GPU {GPU} | {os.cpu_count()} vCPUs")

# ---- Code: local copy, checksums, same code in every track ------------------
assert os.path.isdir(CODE_SRC), f"copy the repository folder code/ to {CODE_SRC}"
shutil.rmtree(CODE, ignore_errors=True)
shutil.copytree(CODE_SRC, CODE)


def sha256(path, chunk=1 << 22):
    """SHA-256 of a file, streamed in 4 MB chunks."""
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(chunk), b""):
            h.update(block)
    return h.hexdigest()


lines = []
for root, dirs, files in os.walk(CODE):
    dirs[:] = sorted(d for d in dirs if not d.startswith(("outputs", "__pycache__", ".")))
    for fn in sorted(files):
        lines.append(f"{sha256(os.path.join(root, fn))}  {os.path.relpath(os.path.join(root, fn), CODE)}")
open(f"{ENV}/code_sha256.txt", "w").write("\n".join(lines) + "\n")
# Only the pipeline code must be identical across tracks and sessions.  Excluded:
# the notebooks in colab/ (Colab rewrites them in Drive when it saves their
# outputs) and hidden files such as .DS_Store, which macOS adds to uploads.
def pipeline(ls):
    keep = []
    for l in ls:
        path = l.split("  ", 1)[1]
        if not path.startswith("colab/") and not os.path.basename(path).startswith("."):
            keep.append(l)
    return keep
PIPELINE = pipeline(lines)
CODE_HASH = hashlib.sha256("\n".join(PIPELINE).encode()).hexdigest()
earlier = sorted((p for p in glob.glob(f"{RUN}/env/*/code_sha256.txt") if not p.startswith(ENV)),
                 key=lambda p: p.split("_session_")[-1])     # earliest session first
if earlier:
    ref = pipeline(open(earlier[0]).read().splitlines())
    changed = sorted(set(ref) ^ set(PIPELINE))
    assert not changed, ("the pipeline code differs from the one this run started with; "
                         "use a new RUN_ID. Differences: %s" % changed[:6])
print("pipeline code hash", CODE_HASH[:16], "| files", len(PIPELINE),
      "| identical to the first session of this run" if earlier else "| first session of this run")

# ---- Dependencies and environment record ------------------------------------
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", f"{CODE}/requirements.txt",
                "transformers", "accelerate"], check=True)


def capture(cmd, dest):
    """Save the output of a shell command into the environment record."""
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    open(f"{ENV}/{dest}", "w").write(r.stdout + r.stderr)


capture(f"{sys.executable} -m pip freeze", "pip_freeze.txt")
capture("nvidia-smi", "nvidia-smi.txt")
capture("lscpu", "lscpu.txt")
capture("free -h; df -h /content", "memory_disk.txt")
import numpy, scipy, sklearn, torchxrayvision, transformers
# The versions pinned with "==" in requirements.txt must be the ones loaded: the
# shuffled cross-validation folds depend on the scikit-learn version (run01).
import re as _re
pinned = dict(_re.findall(r"^([A-Za-z0-9_-]+)==([0-9.]+)", open(f"{CODE}/requirements.txt").read(), _re.M))
loaded = {"numpy": numpy.__version__, "scipy": scipy.__version__, "scikit-learn": sklearn.__version__,
          "torchxrayvision": torchxrayvision.__version__}
wrong = {k: (v, loaded[k]) for k, v in pinned.items() if k in loaded and loaded[k] != v}
assert not wrong, ("pinned vs loaded versions differ %s: Runtime > Restart session, "
                   "then run all cells again" % wrong)
json.dump({"python": sys.version, "platform": platform.platform(), "torch": torch.__version__,
           "cuda": torch.version.cuda, "cudnn": torch.backends.cudnn.version(), "gpu": GPU,
           "vcpus": os.cpu_count(), "torchxrayvision": torchxrayvision.__version__,
           "transformers": transformers.__version__, "numpy": numpy.__version__,
           "scipy": scipy.__version__, "scikit-learn": sklearn.__version__,
           "code_hash": CODE_HASH, "track": TRACK, "session_utc": SESSION},
          open(f"{ENV}/versions.json", "w"), indent=1)

# ---- Job runner ---------------------------------------------------------------


def job(name):
    """Output folder of a job (each job writes only to its own folder)."""
    path = f"{JOBS}/{name}"
    os.makedirs(path, exist_ok=True)
    return path


def run(name, args):
    """Run `python -u <args>` inside CODE; stream and log the output to
    logs/<name>.log and write logs/<name>.done on success.  A job with a .done
    marker is skipped, so the notebook can be re-run after a disconnection."""
    done = f"{LOGS}/{name}.done"
    if os.path.exists(done):
        print(f"[{name}] already done: {open(done).read().strip()}")
        return
    cmd = [sys.executable, "-u"] + [str(x) for x in args]
    env = dict(os.environ, CUBLAS_WORKSPACE_CONFIG=":4096:8", PYTHONUNBUFFERED="1",
               PYTORCH_CUDA_ALLOC_CONF="expandable_segments:True")   # less GPU memory fragmentation
    t0 = time.time()
    with open(f"{LOGS}/{name}.log", "a", encoding="utf8") as log:
        log.write(f"\n===== {utc_now()} track={TRACK} session={SESSION} gpu={GPU}\n$ {shlex.join(cmd)}\n")
        p = subprocess.Popen(cmd, cwd=CODE, env=env, stdout=subprocess.PIPE,
                             stderr=subprocess.STDOUT, text=True, bufsize=1)
        for line in p.stdout:
            print(line, end="")
            log.write(line)
        rc = p.wait()
        log.write(f"===== exit={rc} elapsed={time.time() - t0:.0f}s\n")
    if rc:
        raise RuntimeError(f"{name} failed (exit {rc}); see {LOGS}/{name}.log")
    json.dump({"finished_utc": utc_now(), "elapsed_s": round(time.time() - t0), "track": TRACK,
               "session": SESSION, "gpu": GPU, "code_hash": CODE_HASH, "cmd": cmd}, open(done, "w"))


def copy_inputs(dst_job, src_job, files):
    """Give a job the inputs produced by an earlier job (e.g. manifest, predictions)."""
    for fn in files:
        if not os.path.exists(f"{job(dst_job)}/{fn}"):
            shutil.copy2(f"{job(src_job)}/{fn}", f"{job(dst_job)}/{fn}")


def stage(job_name, *args):
    """Arguments of run_all.py for a job folder."""
    return ["run_all.py", "--nih-dir", NIH, "--out", job(job_name), "--workers", WORKERS,
            "--device", DEVICE] + list(args)

## 3. Tests

In [ ]:
run(f"{TRACK}_tests", ["-m", "pytest", "tests", "-q", "-p", "no:cacheprovider"])

## 4. Data: metadata check, sample, image extraction (sample + boxes + official test split)

In [ ]:
# ---- Metadata: copy from Drive and check against the provenance manifest ------
manifest = json.load(open(f"{DATA_DIR}/DATA_MANIFEST.json"))
for fn, rec in manifest["metadata_files"].items():
    shutil.copy2(f"{DATA_DIR}/meta/{fn}", f"{NIH}/{fn}")
    assert sha256(f"{NIH}/{fn}") == rec["sha256"], f"{fn} differs from DATA_MANIFEST.json"
print("metadata verified:", sorted(manifest["metadata_files"]))

# The sample is deterministic (seed 42) and takes seconds: compute it here to
# know which images to extract.  Every job recomputes it in its own folder and
# the merge checks that all manifests are identical.
run(f"{TRACK}0_sampling", stage(f"{TRACK}0_sampling", "--stages", "sampling"))
import pandas as pd
sample = set(pd.read_csv(f"{job(TRACK + '0_sampling')}/sample_manifest.csv")["Image Index"])
r = json.load(open(f"{job(TRACK + '0_sampling')}/results.json"))
print("sample.n =", r["sample.n"])
assert int(r["sample.n"]) == 50208, "the sample differs from the published one (50,208 images)"


def prepare_images(names):
    """Extract the given images from the archive in Drive into NIH (skips present ones).
    The 45 GB archive is first copied to the local disk: one sequential read from
    Drive is much faster than random access to 50,000 members over the Drive mount."""
    have = {os.path.basename(p) for p in glob.glob(f"{NIH}/**/*.png", recursive=True)}
    need = set(names) - have
    print(f"{len(names)} images needed, {len(need)} to extract")
    if not need:
        return
    assert shutil.disk_usage("/content").free / 1e9 > 75, "need ~75 GB free in /content"
    if not os.path.exists(LOCAL_ZIP):
        t0 = time.time()
        shutil.copyfile(f"{DATA_DIR}/data.zip", LOCAL_ZIP)
        print(f"archive copied from Drive in {(time.time() - t0) / 60:.1f} min")
    with zipfile.ZipFile(LOCAL_ZIP) as zf:
        members = [m for m in zf.infolist() if os.path.basename(m.filename) in need]
        for i, m in enumerate(members):
            dst = f"{NIH}/{m.filename}"
            os.makedirs(os.path.dirname(dst), exist_ok=True)
            with zf.open(m) as src, open(dst, "wb") as out:
                shutil.copyfileobj(src, out)
            if i % 10000 == 0:
                print(f"  {i}/{len(members)}")
    have = {os.path.basename(p) for p in glob.glob(f"{NIH}/**/*.png", recursive=True)}
    missing = set(names) - have
    assert not missing, f"{len(missing)} images missing, e.g. {sorted(missing)[:3]}"
    os.remove(LOCAL_ZIP)                               # free the local disk
    print("images ready; free disk", round(shutil.disk_usage("/content").free / 1e9), "GB")

boxes = set(pd.read_csv(f"{NIH}/BBox_List_2017.csv")["Image Index"])
official = set(open(f"{NIH}/test_list.txt").read().split())
prepare_images(sample | boxes | official)

## 5. Job B1: explainability on the GPU (first validation of the GPU path, T21)

In [ ]:
run("B1_explain", stage("B1_explain", "--stages", "explain"))

## 6. Job B2: predictions of the other models (one image pass per input size)

In [ ]:
run("B2_predict_models", stage("B2_predict_models", "--stages", "sampling,predict",
                               "--models", "densenet_multi,resnet50_multi," + EXTERNAL))

## 7. Job B3: robustness

In [ ]:
run("B3_robustness", stage("B3_robustness", "--stages", "sampling,robustness"))

## 8. Job B4: official NIH test split

In [ ]:
run("B4_official", stage("B4_official", "--stages", "official", "--models", "densenet_nih," + EXTERNAL))

## 9. Job B5: processing time (alone on this machine)

Nothing else may run in this session while it executes. Times are per item on this GPU; the
article's Fig. 10 keeps the published CPU times and reports these side by side.

In [ ]:
run("B5_timing", stage("B5_timing", "--stages", "sampling,timing", "--llm-backend", "local",
                       "--llm-model", LLM_PUBLISHED, "--llm-max-tokens", 1024))
print("Track B finished. When track A is also finished, run 03_merge.ipynb.")